# IPL Sports Analytics - Visualizations Notebook

**Project 11: Is the Player Really Better?**

This notebook creates interactive visualizations for the statistical analysis:
1. Strike rate distributions
2. Regression to mean plots
3. Shrinkage analysis visualizations
4. Uncertainty bars and confidence intervals
5. Player comparison visualizations

In [ ]:
# Setup and imports
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname('.'), 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

from statistics.mean_median import MeanMedianAnalyzer
from statistics.bootstrap_ci import BootstrapCI
from statistics.regression_to_mean import RegressionToMean
from statistics.beta_binomial import BetaBinomialShrinkage
from statistics.player_comparison import PlayerComparisonTool
from statistics.data_access import (
    get_batter_career, get_batter_innings,
    get_batter_career_balls_boundaries, get_boundary_rate_by_player
)

# Set style and random seed
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

print("Libraries loaded successfully")

## 1. Strike Rate Distribution Analysis

Visualizing the distribution of career strike rates across players, showing:
- Mean vs median comparison
- Skewness and kurtosis
- Top performers

In [ ]:
# Initialize analyzer and get data
analyzer = MeanMedianAnalyzer(min_balls_per_innings=10)

# Get career strike rate data
career_sr = analyzer.analyze_career_strike_rates(min_career_balls=200)
career_df = get_batter_career(min_balls=200)

# Create figure with subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Strike Rate Distribution (Career)',
        'Boundary Rate Distribution',
        'Batting Average Distribution',
        'Strike Rate CDF Comparison'
    )
)

# 1. Career Strike Rate Histogram
sr_values = career_df['strike_rate'].dropna()
fig.add_trace(
    go.Histogram(x=sr_values, nbinsx=40, name='SR Count', marker_color='steelblue', showlegend=False),
    row=1, col=1
)
fig.add_vline(x=career_sr['mean'], line_dash='dash', line_color='red', line_width=2, row=1, col=1, annotations=dict(text=f"Mean: {career_sr['mean']:.1f}", font=dict(color='red')))
fig.add_vline(x=career_sr['median'], line_dash='dot', line_color='green', line_width=2, row=1, col=1, annotations=dict(text=f"Median: {career_sr['median']:.1f}", font=dict(color='green')))

# 2. Boundary Rate Histogram
br_values = career_df['boundary_rate'].dropna()
fig.add_trace(
    go.Histogram(x=br_values, nbinsx=40, name='BR Count', marker_color='coral', showlegend=False),
    row=1, col=2
)

# 3. Batting Average Histogram
avg_df = career_df[career_df['dismissals'] >= 5].copy()
avg_df['average'] = avg_df['runs'] / avg_df['dismissals']
avg_values = avg_df['average'].dropna()
fig.add_trace(
    go.Histogram(x=avg_values, nbinsx=40, name='Avg Count', marker_color='mediumseagreen', showlegend=False),
    row=2, col=1
)

# 4. CDF Comparison - Top 10 players
top_players = career_df.nlargest(10, 'strike_rate')['player_name'].tolist()
for player in top_players:
    innings_df = get_batter_innings(player, min_balls=10)
    if len(innings_df) > 0:
        fig.add_trace(
            go.Scatter(x=sorted(innings_df['strike_rate']), y=np.arange(1, len(innings_df)+1)/len(innings_df),
                       mode='lines', name=player, showlegend=False, line=dict(width=2)),
            row=2, col=2
        )

fig.update_layout(height=800, width=1200, title_text='Strike Rate Analyses', title_x=0.5)
fig.update_xaxes(title_text='Strike Rate', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_xaxes(title_text='Boundary Rate (%)', row=1, col=2)
fig.update_yaxes(title_text='Count', row=1, col=2)
fig.update_xaxes(title_text='Batting Average', row=2, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.update_xaxes(title_text='Strike Rate', row=2, col=2)
fig.update_yaxes(title_text='CDF', row=2, col=2)

fig.show()

# Print statistics
print(f"\nStrike Rate Statistics:")
print(f"  Mean: {career_sr['mean']:.2f}")
print(f"  Median: {career_sr['median']:.2f}")
print(f"  Skewness: {career_sr['skewness']:.4f}")
print(f"  Kurtosis: {career_sr['kurtosis']:.4f}")
print(f"  N players (min 200 balls): {career_sr['n_observations']}")

## 2. Bootstrap Confidence Interval Visualizations

Visualizing bootstrap distributions and confidence intervals for player strike rates, boundary rates, and batting averages, showing uncertainty intervals.

In [ ]:
# Initialize bootstrap with production settings
bootstrap = BootstrapCI(n_bootstrap=10000, confidence_level=0.95, min_balls_per_innings=10)

# Top players by career balls faced
top_players = ['V Kohli', 'RG Sharma', 'CH Gayle', 'AB de Villiers', 'DA Warner', 'MS Dhoni', 'KL Rahul', 'S Dhawan']

# Collect bootstrap results
results = []
for player in top_players:
    result = bootstrap.player_career_sr_ci(player)
    results.append({
        'player': player,
        'estimate': result.original_estimate,
        'ci_lower': result.ci_lower,
        'ci_upper': result.ci_upper,
        'ci_width': result.ci_upper - result.ci_lower,
        'bootstrap_dist': result.bootstrap_distribution
    })

# Create visualization
fig = go.Figure()

for i, r in enumerate(results):
    # Add bootstrap distribution as scatter
    fig.add_trace(go.Scatter(
        x=r['bootstrap_dist'][::10],  # Subsample for clarity
        y=[i] * len(r['bootstrap_dist'][::10]),
        mode='markers', marker=dict(size=3, color='blue', opacity=0.3),
        showlegend=False, hoverinfo='skip'
    ))
    
    # Add point estimate and CI
    fig.add_trace(go.Scatter(
        x=[r['ci_lower'], r['estimate'], r['ci_upper']],
        y=[i, i, i],
        mode='markers', marker=dict(size=12, color='red'),
        name=f"{r['player']}\nSR={r['estimate']:.1f}% CI=[{r['ci_lower']:.1f}, {r['ci_upper']:.1f}]",
        error_x=dict(type='data', symmetric=False, arrayminus=[r['estimate']-r['ci_lower']]*len(results), array=[r['ci_upper']-r['estimate']]*len(results)),
        hovertemplate=f"{r['player']}<br>SR: {r['estimate']:.1f}<br>95% CI: [{r['ci_lower']:.1f}, {r['ci_upper']:.1f}]<extra></extra>"
    ))

fig.update_yaxes(tickmode='array', tickvals=list(range(len(results))), ticktext=[r['player'] for r in results], title='Player')
fig.update_xaxes(title='Strike Rate (percentile bootstrap CI)')
fig.update_layout(height=400, width=900, title_text='Bootstrap Confidence Intervals - Career Strike Rate', title_x=0.5, showlegend=False, hovermode='y')
fig.show()

# Create bar chart showing CI widths
ci_df = pd.DataFrame([{'player': r['player'], 'estimate': r['estimate'], 'width': r['ci_width']} for r in results])
fig2 = px.bar(ci_df, x='player', y='width', text=ci_df['width'].round(1).astype(str),
              title='CI Width by Player (Wider = More Uncertainty)',
              color='width', color_continuous_scale='Reds')
fig2.update_traces(textposition='outside', rotation=45)
fig2.update_layout(width=900, height=400)
fig2.show()

## 3. Regression to Mean Visualization

Primary visualization: First-half vs second-half strike rates showing regression to the mean effect. Players should regress toward the league average if second-half performance is just sampling variation.

In [ ]:
# Initialize regression analyzer
regression = RegressionToMean(min_balls_per_half=100)

# Get all player results
results = regression.analyze_all_players()
theory_exp = regression.theory_vs_experiment()
over, under = regression.identify_overperformers_underperformers(threshold_pct=10)

# Create main regression plot
fig = go.Figure()

# Add all players as scatter
for r in results:
    color = 'green' if r.regression_magnitude > 10 else ('red' if r.regression_magnitude < -10 else 'gray')
    fig.add_trace(go.Scatter(
        x=[r.first_half_sr],
        y=[r.second_half_sr],
        mode='markers+text',
        marker=dict(size=8, color=color, opacity=0.7),
        text=[r.player_name[:15] if len(r.player_name) > 15 else r.player_name],
        textposition='middle center',
        name=r.player_name,
        showlegend=False,
        hovertemplate=f"{r.player_name}<br>First Half SR: {r.first_half_sr:.1f}<br>Second Half SR: {r.second_half_sr:.1f}<br>Regression: {r.regression_magnitude:+.1f}<extra></extra>"
    ))

# Add identity line (no regression)
max_sr = max(max(r.first_half_sr for r in results), max(r.second_half_sr for r in results), 180)
fig.add_trace(go.Scatter(
    x=[0, max_sr], y=[0, max_sr],
    mode='lines', line=dict(color='gray', dash='dash'),
    name='No Regression (y=x)',
    showlegend=True
))

# Add regression line
x_vals = np.array([r.first_half_sr for r in results])
y_vals = np.array([r.second_half_sr for r in results])
slope, intercept = np.polyfit(x_vals, y_vals, 1)
x_line = np.linspace(80, max_sr, 100)
y_line = slope * x_line + intercept
fig.add_trace(go.Scatter(
    x=x_line, y=y_line,
    mode='lines', line=dict(color='red', width=2),
    name=f'Regression Line (slope={slope:.3f})',
    showlegend=True
))

fig.update_layout(
    title='Regression to Mean: First Half vs Second Half Strike Rate',
    xaxis_title='First Half Career Strike Rate',
    yaxis_title='Second Half Career Strike Rate',
    width=700, height=650,
    template='plotly_white',
    shapes=[dict(line=dict(color='lightcoral', width=1.5), type='line', x0=0, x1=max_sr, y0=max_sr, y1=0)]
)
fig.show()

# Create highlights table for over/under performers
print("\n=== OVERPERFORMERS (Regression < -10) ===")
print("Players whose 2nd half was BETTER than expected by regression")
for r in sorted(over, key=lambda x: x.regression_magnitude)[:5]:
    print(f"  {r.player_name}: {r.first_half_sr:.1f} -> {r.second_half_sr:.1f} (change: {r.regression_magnitude:+.1f})")

print("\n=== UNDERPERFORMERS (Regression > +10) ===")
print("Players whose 2nd half was WORSE than expected by regression")
for r in sorted(under, key=lambda x: x.regression_magnitude, reverse=True)[:5]:
    print(f"  {r.player_name}: {r.first_half_sr:.1f} -> {r.second_half_sr:.1f} (change: {r.regression_magnitude:+.1f})")

## 4. Beta-Binomial Shrinkage Visualizations

Visualizing how sample size affects shrinkage toward the league average. Players with more balls faced should have less shrinkage.

In [ ]:
# Initialize shrinkage analyzer
shrinkage = BetaBinomialShrinkage(min_balls=200)

# Get all player results
df, prior = shrinkage.analyze_all_players()

# 1. Raw vs Shrunken scatter plot
fig1 = px.scatter(df, x='raw_boundary_rate', y='shrunken_boundary_rate', 
                    hover_data=['player_name', 'balls_faced', 'boundary_rate'],
                    title='Raw vs Shrunken Boundary Rate (Shrinkage toward League Avg)',
                    labels={'raw_boundary_rate': 'Raw Boundary Rate (%)', 'shrunken_boundary_rate': 'Shrunken Boundary Rate (%)'})
fig1.add_shape(type='line', x0=df['raw_boundary_rate'].min(), y0=df['raw_boundary_rate'].min(),
               x1=df['raw_boundary_rate'].max(), y1=df['raw_boundary_rate'].max(),
               line=dict(surface='gray', dash='dash'))
fig1.update_layout(width=700, height=500)
fig1.show()

# 2. Shrinkage factor vs balls faced
fig2 = px.scatter(df, x='balls_faced', y='shrinkage_factor', 
                  hover_data=['player_name', 'raw_boundary_rate', 'shrunken_boundary_rate'],
                  title='Shrinkage Factor vs Sample Size (More balls = Less shrinkage)')

# Add trend line
from scipy.stats import spearmanr
corr, p_val = spearmanr(df['balls_faced'], df['shrinkage_factor'])
fig2.add_trace(go.Scatter(
    x=df['balls_faced'],
    y=df['shrinkage_factor'].rolling(50, center=True).mean(),
    mode='lines', line=dict(color='red', width=3), name='Smoothed'
))
fig2.update_traces(marker=dict(size=8, opacity=0.6))
fig2.update_xaxes(title='Balls Faced (Log scale)', type='log')
fig2.update_layout(width=700, height=500, annotations=dict(text=f"Spearman ρ={corr:.3f}, p={p_val:.3f}"))
fig2.show()

# 3. Distribution of raw vs shrunken rates
fig3 = go.Figure()
fig3.add_trace(go.Histogram(x=df['raw_boundary_rate'], name='Raw', opacity=0.6, nbinsx=40, marker_color='blue'))
fig3.add_trace(go.Histogram(x=df['shrunken_boundary_rate'], name='Shrunken', opacity=0.6, nbinsx=40, marker_color='red'))
fig3.update_layout(title='Distribution of Raw vs Shrunken Boundary Rates (Shrinkage reduces variance)',
                   xaxis_title='Boundary Rate (%)', yaxis_title='Count', barmode='overlay')
fig3.show()

print(f"\nPrior Parameters (Method of Moments):")
print(f"  Alpha: {prior.alpha:.2f}")
print(f"  Beta: {prior.beta:.2f}")
print(f"  League Boundary Rate: {prior.league_boundary_rate*100:.2f}%")
print(f"  Prior Sample Size (α+β): {prior.alpha + prior.beta:.0f} balls")

## 5. Player Comparison Tool - Uncertainty Visualization

Creating uncertainty bars and confidence intervals for player comparisons.

In [ ]:
# Initialize comparison tool
tool = PlayerComparisonTool(min_balls_per_innings=10, min_career_balls=200)

# Compare two top players
p1, p2 = 'V Kohli', 'RG Sharma'
result = tool.compare(p1, p2, 'strike_rate')

# Get career data for CIs
bootstrap = BootstrapCI(n_bootstrap=10000, confidence_level=0.95, min_balls_per_innings=10)
ci1 = bootstrap.player_career_sr_ci(p1)
ci2 = bootstrap.player_career_sr_ci(p2)
diff_ci = bootstrap.compare_players_ci(p1, p2, 'strike_rate')

# Create waterfall-style comparison with uncertainty
fig = go.Figure()

# Add bars for each player with CI error bars
fig.add_trace(go.Bar(
    name=p1,
    x=[p1],
    y=[ci1.original_estimate],
    error_y=dict(type='data', symmetric=False, arrayplus=[ci1.ci_upper - ci1.original_estimate], arrayminus=[ci1.original_estimate - ci1.ci_lower]),
    marker_color='steelblue'
))

fig.add_trace(go.Bar(
    name=p2,
    x=[p2],
    y=[ci2.original_estimate],
    error_y=dict(type='data', symmetric=False, arrayplus=[ci2.ci_upper - ci2.original_estimate], arrayminus=[ci2.original_estimate - ci2.ci_lower]),
    marker_color='coral'
))

fig.update_layout(
    title=f'{p1} vs {p2} - Career Strike Rate Comparison with 95% CI',
    yaxis_title='Strike Rate',
    xaxis_title='Player',
    width=700, height=500,
    template='plotly_white',
    showlegend=True
)
fig.show()

print(f"\nComparison Results:")
print(f"  Verdict: {result.verdict}")
print(f"  Confidence: {result.confidence}")
print(f"  Difference CI: [{diff_ci['ci_lower']:.2f}, {diff_ci['ci_upper']:.2f}]")
print(f"  CI Excludes Zero: {diff_ci['excludes_zero']} (significant if True)")
print(f"\nKey Findings:")
for finding in result.key_findings:
    print(f"  - {finding}")

## 6. Multiple Comparisons - P-value Distribution

Visualizing how multiple testing corrections affect significance decisions.

In [ ]:
from statistics.multiple_comparisons import MultipleComparisons

# Compare Kohli vs 10 other batters
mc = MultipleComparisons(alpha=0.05)
comparison_players = ['RG Sharma', 'CH Gayle', 'AB de Villiers', 'MS Dhoni', 'DA Warner', 
                      'KL Rahul', 'S Dhawan', 'AM Rahane', 'JC Buttler', 'HH Pandya']

results = mc.player_comparison_correction('V Kohli', comparison_players, 'strike_rate', min_balls=10)

# Extract p-values for visualization
names = [r['test'].split(' vs ')[0] for r in results['holm_bonferroni']['results']]
raw_p = [r['raw_p'] for r in results['holm_bonferroni']['results']]
holm_p = [r['corrected_p'] for r in results['holm_bonferroni']['results']]
bh_p = [r['corrected_p'] for r in results['benjamini_hochberg']['results']]
bonf_p = [r['corrected_p'] for r in results['bonferroni']['results']]

# Create p-value comparison plot
fig = go.Figure()

y_pos = np.arange(len(names))
fig.add_trace(go.Scatter(y=[i + 0.3 for i in y_pos], x=raw_p, mode='markers', name='Raw p-value', marker=dict(size=10), error_y=dict(type='data', array=[0]*len(names))))
fig.add_trace(go.Scatter(y=[i + 0.2 for i in y_pos], x=holm_p, mode='markers', name='Holm-Bonferroni', marker=dict(size=10)))
fig.add_trace(go.Scatter(y=[i + 0.1 for i in y_pos], x=bh_p, mode='markers', name='Benjamini-Hochberg', marker=dict(size=10)))
fig.add_trace(go.Scatter(y=[i for i in y_pos], x=bonf_p, mode='markers', name='Bonferroni', marker=dict(size=10)))

fig.add_vline(x=0.05, line_dash='red', line_width=2, annotation_text='α=0.05')

fig.update_yaxes(tickmode='array', tickvals=y_pos, ticktext=names, title='Player Compared')
fig.update_xaxes(title='P-value (log scale)', type='log')
fig.update_layout(title='Multiple Comparison Corrections: Raw vs Adjusted P-values', height=600, width=900, template='plotly_white')
fig.show()

print(f"\nSummary of Corrections:")
print(f"  Raw p<0.05: {sum(1 for p in raw_p if p < 0.05)}/{len(raw_p)}")
print(f"  Holm-Bonferroni: {sum(1 for p in holm_p if p < 0.05)}/{len(holm_p)} rejected")
print(f"  Benjamini-Hochberg: {sum(1 for p in bh_p if p < 0.05)}/{len(bh_p)} rejected")
print(f"  Bonferroni: {sum(1 for p in bonf_p if p < 0.05)}/{len(bonf_p)} rejected")

## 7. Summary Dashboard

A summary of all key visualizations and findings.

In [ ]:
# Create a comprehensive summary dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Top Players by Career SR', 'CI Width Distribution', 'Shrinkage Correlation', 'Regression Analysis'),
    specs=[[{'type': 'bar'}, {'type': 'histogram'}], [{'type': 'scatter'}, {'type': 'scatter'}]]
)

# 1. Top 10 players by career SR
top10 = career_df.nlargest(10, 'strike_rate')
fig.add_trace(go.Bar(x=top10['player_name'], y=top10['strike_rate'], marker_color='steelblue'), row=1, col=1)

# 2. CI width distribution
ci_df = pd.DataFrame([{'player': r['player'], 'estimate': r['estimate'], 'width': r['ci_width']} for r in results])
fig.add_trace(go.Histogram(x=ci_df['width'], nbinsx=20, marker_color='coral'), row=1, col=2)

# 3. Shrinkage correlation plot
fig.add_trace(go.Scatter(x=df['raw_boundary_rate'], y=df['shrunken_boundary_rate'], mode='markers', 
                         marker=dict(size=5, opacity=0.5), name='Boundary Rate', showlegend=False), row=2, col=1)

# 4. Regression plot (subset)
sample_results = results[:50]
fig.add_trace(go.Scatter(x=[r.first_half_sr for r in sample_results], y=[r.second_half_sr for r in sample_results],
                         mode='markers', marker=dict(size=8, opacity=0.6), name='Regression', showlegend=False), row=2, col=2)

fig.update_xaxes(title_text='Strike Rate', row=1, col=1)
fig.update_xaxes(title_text='CI Width', row=1, col=2)
fig.update_xaxes(title_text='Raw Boundary Rate', row=2, col=1)
fig.update_xaxes(title_text='First Half SR', row=2, col=2)
fig.update_yaxes(title_text='Strike Rate', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=2)
fig.update_yaxes(title_text='Shrunken Rate', row=2, col=1)
fig.update_yaxes(title_text='Second Half SR', row=2, col=2)
fig.update_layout(height=800, width=1200, title_text='Statistical Analysis Summary Dashboard', title_x=0.5)
fig.show()

print("\n" + "="*60)
print("VISUALIZATION SUMMARY - Key Insights")
print("="*60)
print(f"1. Strike Rate Distribution: Mean={career_sr['mean']:.1f}, Median={career_sr['median']:.1f}, Skew={career_sr['skewness']:.2f}")
print(f"2. Top Player: {top10.iloc[0]['player_name']} with SR {top10.iloc[0]['strike_rate']:.1f}")
print(f"3. CI widths range: [{ci_df['width'].min():.1f}, {ci_df['width'].max():.1f}]")
print(f"4. Shrinkage correlation: {df['raw_boundary_rate'].corr(df['shrunken_boundary_rate']):.3f}")
print(f"5. Regression slope: {slope:.3f} (theory suggests < 1)")
print("="*60)